# Notebook 07 — Industrialisation du pipeline de Machine Learning

## Objectif

Les notebooks précédents ont permis de préparer les données, entraîner plusieurs modèles, sélectionner une régression logistique finale et sauvegarder les artefacts nécessaires à la prédiction.

Ce notebook ne réentraîne pas le modèle. Il vérifie que le système final peut être utilisé comme un véritable pipeline de prédiction :

- chargement du modèle, du scaler et des paramètres de preprocessing ;
- validation des données brutes entrantes ;
- reproduction automatique du nettoyage, du feature engineering et des encodages ;
- traitement des valeurs `None` ou `NaN` ;
- prédiction d'un passager ou d'un lot de passagers ;
- contrôle des erreurs de structure ;
- exécution des tests automatisés.

## 1. Architecture utilisée

Le pipeline final suit la chaîne suivante :

-   Données brutes d'un passager
-   validate_input()
-   prepare_new_data_for_prediction()
-   Imputation avec les statistiques du train
-   Feature engineering et One-Hot Encoding
-   Alignement sur les 15 features finales  
-   StandardScaler déjà entraîné
-   Régression logistique finale
-   Prédiction et probabilité de survie


Le code métier est conservé dans `src/preprocessing.py` et `src/predict.py`. Le notebook sert uniquement à charger, utiliser, vérifier et documenter ce code.

## 2. Imports et détection de la racine du projet

In [1]:
from pathlib import Path
import sys
import subprocess

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


def find_project_root(start: Path) -> Path:
    """Recherche le dossier contenant simultanément src/, models/ et tests/."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if all((candidate / folder).exists() for folder in ["src", "models", "tests"]):
            return candidate

    raise FileNotFoundError(
        "Impossible de trouver la racine du projet contenant src/, models/ et tests/."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

MODEL_DIR = PROJECT_ROOT / "models"
TEST_DIR = PROJECT_ROOT / "tests"
SRC_DIR = PROJECT_ROOT / "src"

print("Racine du projet :", PROJECT_ROOT)
print("Dossier models    :", MODEL_DIR)
print("Dossier src       :", SRC_DIR)
print("Dossier tests     :", TEST_DIR)

Racine du projet : /Users/othmanbenmoussa/Desktop/Titanic_Classification
Dossier models    : /Users/othmanbenmoussa/Desktop/Titanic_Classification/models
Dossier src       : /Users/othmanbenmoussa/Desktop/Titanic_Classification/src
Dossier tests     : /Users/othmanbenmoussa/Desktop/Titanic_Classification/tests


## 3. Import des fonctions du pipeline

In [2]:
from src.preprocessing import (
    prepare_new_data_for_prediction,
    validate_input,
)
from src.predict import predict

print("✓ Fonctions de preprocessing importées")
print("✓ Fonction de prédiction importée")

✓ Fonctions de preprocessing importées
✓ Fonction de prédiction importée


## 4. Chargement des artefacts

In [3]:
SCALER_PATH = MODEL_DIR / "standard_scaler.pkl"
MODEL_PATH = MODEL_DIR / "logistic_regression_final.pkl"
PREPROCESSING_PATH = MODEL_DIR / "preprocessing_info.pkl"

required_artifacts = [
    SCALER_PATH,
    MODEL_PATH,
    PREPROCESSING_PATH,
]

missing_artifacts = [
    str(path)
    for path in required_artifacts
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Artefacts manquants : - " + "- ".join(missing_artifacts)
    )

scaler = joblib.load(SCALER_PATH)
model = joblib.load(MODEL_PATH)
preprocessing_info = joblib.load(PREPROCESSING_PATH)

training_medians = preprocessing_info["training_medians"]
training_modes = preprocessing_info["training_modes"]

print("✓ Scaler chargé")
print("✓ Modèle chargé")
print("✓ Paramètres de preprocessing chargés")

✓ Scaler chargé
✓ Modèle chargé
✓ Paramètres de preprocessing chargés


## 5. Vérification des artefacts

In [4]:
FEATURE_COLUMNS = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Cabin_known",
    "FamilySize",
    "IsAlone",
    "Embarked_Q",
    "Embarked_S",
    "Title_Miss",
    "Title_Mr",
    "Title_Mrs",
    "Title_Rare",
]

print("Type du scaler :", type(scaler))
print("Type du modèle :", type(model))
print("Nombre de features du scaler :", scaler.n_features_in_)
print("Nombre de features du modèle :", model.n_features_in_)
print("Médianes d'entraînement :", training_medians)
print("Modes d'entraînement :", training_modes)

assert scaler.n_features_in_ == len(FEATURE_COLUMNS)
assert model.n_features_in_ == len(FEATURE_COLUMNS)

if hasattr(scaler, "feature_names_in_"):
    assert scaler.feature_names_in_.tolist() == FEATURE_COLUMNS

print("✓ Les artefacts sont cohérents avec les 15 features attendues")

Type du scaler : <class 'sklearn.preprocessing._data.StandardScaler'>
Type du modèle : <class 'sklearn.linear_model._logistic.LogisticRegression'>
Nombre de features du scaler : 15
Nombre de features du modèle : 15
Médianes d'entraînement : {'Pclass': np.float64(3.0), 'Age': np.float64(28.5), 'SibSp': np.float64(0.0), 'Parch': np.float64(0.0), 'Fare': np.float64(14.4542)}
Modes d'entraînement : {'Sex': 'male', 'Embarked': 'S'}
✓ Les artefacts sont cohérents avec les 15 features attendues


## 6. Scénario A — Passager complet

Ce premier scénario représente le cas nominal : toutes les informations brutes nécessaires sont renseignées.

In [5]:
passenger_complete = pd.DataFrame([{
    "Pclass": 1,
    "Name": "Martin, Mrs. Sophie",
    "Sex": "female",
    "Age": 28,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 80.0,
    "Cabin": "B20",
    "Embarked": "C",
}])

display(passenger_complete)

,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,1,"Martin, Mrs. Sophie",female,28,0,0,80.0,B20,C


### 6.1 Validation de la structure

In [6]:
validate_input(passenger_complete)
print("✓ Structure valide")

✓ Structure valide


### 6.2 Préparation et standardisation

In [7]:
passenger_complete_scaled = prepare_new_data_for_prediction(
    passenger_complete,
    scaler,
)

print("Shape finale :", passenger_complete_scaled.shape)
print("Valeurs manquantes :", passenger_complete_scaled.isna().sum().sum())
display(passenger_complete_scaled)

ÉTAPE 0 — DONNÉES BRUTES
Shape : (1, 9)
Colonnes : ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']
✓ Survived absent
✓ PassengerId supprimé si présent
✓ Toutes les colonnes brutes obligatoires sont présentes

ÉTAPE 1 — IMPUTATION DES VARIABLES NUMÉRIQUES
✓ Pclass : toutes les valeurs étaient renseignées
✓ Age : toutes les valeurs étaient renseignées
✓ SibSp : toutes les valeurs étaient renseignées
✓ Parch : toutes les valeurs étaient renseignées
✓ Fare : toutes les valeurs étaient renseignées

ÉTAPE 2 — IMPUTATION DES VARIABLES CATÉGORIELLES
✓ Sex : toutes les valeurs étaient renseignées
✓ Embarked : toutes les valeurs étaient renseignées
✓ Name manquant traité comme titre Rare
✓ Cabin manquante conservée pour calculer Cabin_known

ÉTAPE 3 — CRÉATION DES VARIABLES
✓ Cabin_known créée
✓ FamilySize créée
✓ IsAlone créée

Vérification :
   Cabin_known  FamilySize  IsAlone
0            1           1        1

ÉTAPE 4 — CRÉATION DE TITLE
Titres obtenus :
Titl

,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin_known,FamilySize,IsAlone,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,-1.571457,1.346933,-0.112078,-0.465084,-0.466183,1.003224,1.857418,-0.556339,0.800346,-0.289333,-1.634045,-0.503509,-1.171893,2.377857,-0.161048


### 6.3 Prédiction

In [8]:
prediction_complete, probability_complete, _ = predict(
    passenger_complete,
    scaler,
    model,
)

result_complete = pd.DataFrame({
    "prediction": prediction_complete,
    "label": [
        "Survivant" if value == 1 else "Non survivant"
        for value in prediction_complete
    ],
    "survival_probability": probability_complete[:, 1],
})

display(result_complete)

ÉTAPE 0 — DONNÉES BRUTES
Shape : (1, 9)
Colonnes : ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']
✓ Survived absent
✓ PassengerId supprimé si présent
✓ Toutes les colonnes brutes obligatoires sont présentes

ÉTAPE 1 — IMPUTATION DES VARIABLES NUMÉRIQUES
✓ Pclass : toutes les valeurs étaient renseignées
✓ Age : toutes les valeurs étaient renseignées
✓ SibSp : toutes les valeurs étaient renseignées
✓ Parch : toutes les valeurs étaient renseignées
✓ Fare : toutes les valeurs étaient renseignées

ÉTAPE 2 — IMPUTATION DES VARIABLES CATÉGORIELLES
✓ Sex : toutes les valeurs étaient renseignées
✓ Embarked : toutes les valeurs étaient renseignées
✓ Name manquant traité comme titre Rare
✓ Cabin manquante conservée pour calculer Cabin_known

ÉTAPE 3 — CRÉATION DES VARIABLES
✓ Cabin_known créée
✓ FamilySize créée
✓ IsAlone créée

Vérification :
   Cabin_known  FamilySize  IsAlone
0            1           1        1

ÉTAPE 4 — CRÉATION DE TITLE
Titres obtenus :
Titl

,prediction,label,survival_probability
0,1,Survivant,0.979988


## 7. Scénario B — Passager avec valeurs manquantes

Le pipeline doit conserver les valeurs connues et remplacer uniquement les valeurs manquantes :

- variables numériques → médianes du jeu d'entraînement ;
- `Sex` et `Embarked` → modes du jeu d'entraînement ;
- `Cabin=None` → `Cabin_known=0` ;
- `Name=None` → titre classé `Rare`.

In [9]:
passenger_missing = pd.DataFrame([{
    "Pclass": 3,
    "Name": None,
    "Sex": None,
    "Age": None,
    "SibSp": None,
    "Parch": None,
    "Fare": None,
    "Cabin": None,
    "Embarked": None,
}])

display(passenger_missing)

,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,3,None,None,None,None,None,None,None,None


### 7.1 Préparation et prédiction malgré les valeurs manquantes

In [10]:
prediction_missing, probability_missing, passenger_missing_scaled = predict(
    passenger_missing,
    scaler,
    model,
)

result_missing = pd.DataFrame({
    "prediction": prediction_missing,
    "label": [
        "Survivant" if value == 1 else "Non survivant"
        for value in prediction_missing
    ],
    "survival_probability": probability_missing[:, 1],
})

print("Shape standardisée :", passenger_missing_scaled.shape)
print("Valeurs manquantes après preprocessing :", passenger_missing_scaled.isna().sum().sum())
display(passenger_missing_scaled)
display(result_missing)

ÉTAPE 0 — DONNÉES BRUTES
Shape : (1, 9)
Colonnes : ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']
✓ Survived absent
✓ PassengerId supprimé si présent
✓ Toutes les colonnes brutes obligatoires sont présentes

ÉTAPE 1 — IMPUTATION DES VARIABLES NUMÉRIQUES
✓ Pclass : toutes les valeurs étaient renseignées
✓ Age : 1 valeur(s) remplacée(s) par 28.5
✓ SibSp : 1 valeur(s) remplacée(s) par 0.0
✓ Parch : 1 valeur(s) remplacée(s) par 0.0
✓ Fare : 1 valeur(s) remplacée(s) par 14.4542

ÉTAPE 2 — IMPUTATION DES VARIABLES CATÉGORIELLES
✓ Sex : 1 valeur(s) remplacée(s) par 'male'
✓ Embarked : 1 valeur(s) remplacée(s) par 'S'
✓ Name manquant traité comme titre Rare
✓ Cabin manquante conservée pour calculer Cabin_known

ÉTAPE 3 — CRÉATION DES VARIABLES
✓ Cabin_known créée
✓ FamilySize créée
✓ IsAlone créée

Vérification :
   Cabin_known  FamilySize  IsAlone
0            0         1.0        1

ÉTAPE 4 — CRÉATION DE TITLE
Titres obtenus :
Title
Rare    1
Name: count, dty

/Users/othmanbenmoussa/Desktop/Titanic_Classification/src/preprocessing.py:82: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.replace({None: np.nan})
/Users/othmanbenmoussa/Desktop/Titanic_Classification/src/preprocessing.py:324: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X["Title"] = X["Title"].replace({


,Pclass,Sex,Age,SibSp,Parch,Fare,Cabin_known,FamilySize,IsAlone,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0.829568,-0.742427,-0.073691,-0.465084,-0.466183,-0.361593,-0.538382,-0.556339,0.800346,-0.289333,0.611978,-0.503509,-1.171893,-0.420547,6.209312


,prediction,label,survival_probability
0,0,Non survivant,0.149629


## 8. Scénario C — Prédiction par lot

Le pipeline doit accepter plusieurs passagers dans un même DataFrame et retourner une prédiction et une probabilité pour chaque ligne.

In [11]:
passengers_batch = pd.DataFrame([
    {
        "Pclass": 1,
        "Name": "Martin, Miss. Alice",
        "Sex": "female",
        "Age": 22,
        "SibSp": 0,
        "Parch": 0,
        "Fare": 75.0,
        "Cabin": "C85",
        "Embarked": "C",
    },
    {
        "Pclass": 3,
        "Name": "Durand, Mr. Paul",
        "Sex": "male",
        "Age": 38,
        "SibSp": 0,
        "Parch": 0,
        "Fare": 8.05,
        "Cabin": None,
        "Embarked": "S",
    },
    {
        "Pclass": 2,
        "Name": "Example, Mrs. Unknown",
        "Sex": "female",
        "Age": None,
        "SibSp": 1,
        "Parch": 1,
        "Fare": None,
        "Cabin": None,
        "Embarked": None,
    },
])

display(passengers_batch)

,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Cabin,Embarked
0,1,"Martin, Miss. Alice",female,22.0,0,0,75.00,C85,C
1,3,"Durand, Mr. Paul",male,38.0,0,0,8.05,None,S
2,2,"Example, Mrs. Unknown",female,NaN,1,1,NaN,None,None


### 8.1 Résultats du lot

In [12]:
predictions_batch, probabilities_batch, passengers_batch_scaled = predict(
    passengers_batch,
    scaler,
    model,
)

results_batch = passengers_batch[["Name", "Pclass", "Sex", "Age"]].copy()
results_batch["prediction"] = predictions_batch
results_batch["label"] = [
    "Survivant" if value == 1 else "Non survivant"
    for value in predictions_batch
]
results_batch["survival_probability"] = probabilities_batch[:, 1]

assert len(predictions_batch) == len(passengers_batch)
assert probabilities_batch.shape == (len(passengers_batch), 2)
assert passengers_batch_scaled.shape == (len(passengers_batch), 15)

display(results_batch)

ÉTAPE 0 — DONNÉES BRUTES
Shape : (3, 9)
Colonnes : ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']
✓ Survived absent
✓ PassengerId supprimé si présent
✓ Toutes les colonnes brutes obligatoires sont présentes

ÉTAPE 1 — IMPUTATION DES VARIABLES NUMÉRIQUES
✓ Pclass : toutes les valeurs étaient renseignées
✓ Age : 1 valeur(s) remplacée(s) par 28.5
✓ SibSp : toutes les valeurs étaient renseignées
✓ Parch : toutes les valeurs étaient renseignées
✓ Fare : 1 valeur(s) remplacée(s) par 14.4542

ÉTAPE 2 — IMPUTATION DES VARIABLES CATÉGORIELLES
✓ Sex : toutes les valeurs étaient renseignées
✓ Embarked : 1 valeur(s) remplacée(s) par 'S'
✓ Name manquant traité comme titre Rare
✓ Cabin manquante conservée pour calculer Cabin_known

ÉTAPE 3 — CRÉATION DES VARIABLES
✓ Cabin_known créée
✓ FamilySize créée
✓ IsAlone créée

Vérification :
   Cabin_known  FamilySize  IsAlone
0            1           1        1
1            0           1        1
2            0           3 

,Name,Pclass,Sex,Age,prediction,label,survival_probability
0,"Martin, Miss. Alice",1,female,22.0,1,Survivant,0.965275
1,"Durand, Mr. Paul",3,male,38.0,0,Non survivant,0.058505
2,"Example, Mrs. Unknown",2,female,NaN,1,Survivant,0.769655


## 9. Scénario D — Entrée invalide

Une colonne entièrement absente constitue une erreur de structure. Le pipeline doit refuser l'entrée avec un message explicite plutôt que d'inventer silencieusement la variable.

In [ ]:
invalid_passenger = passenger_complete.drop(columns=["Embarked"])

try:
    predict(
        invalid_passenger,
        scaler,
        model,
    )
except ValueError as error:
    print("Erreur correctement détectée :")
    print(error)

## 10. Contrôles finaux du pipeline

In [13]:
assert passenger_complete_scaled.shape == (1, 15)
assert passenger_missing_scaled.shape == (1, 15)
assert passengers_batch_scaled.shape == (3, 15)

assert passenger_complete_scaled.columns.tolist() == FEATURE_COLUMNS
assert passenger_missing_scaled.columns.tolist() == FEATURE_COLUMNS
assert passengers_batch_scaled.columns.tolist() == FEATURE_COLUMNS

assert passenger_complete_scaled.isna().sum().sum() == 0
assert passenger_missing_scaled.isna().sum().sum() == 0
assert passengers_batch_scaled.isna().sum().sum() == 0

assert np.isfinite(passenger_complete_scaled.to_numpy()).all()
assert np.isfinite(passenger_missing_scaled.to_numpy()).all()
assert np.isfinite(passengers_batch_scaled.to_numpy()).all()

assert result_complete["survival_probability"].between(0, 1).all()
assert result_missing["survival_probability"].between(0, 1).all()
assert results_batch["survival_probability"].between(0, 1).all()

print("✓ Dimensions conformes")
print("✓ Colonnes conformes")
print("✓ Aucune valeur manquante")
print("✓ Aucune valeur infinie")
print("✓ Probabilités comprises entre 0 et 1")
print("VALIDATION FONCTIONNELLE DU PIPELINE : OK")

✓ Dimensions conformes
✓ Colonnes conformes
✓ Aucune valeur manquante
✓ Aucune valeur infinie
✓ Probabilités comprises entre 0 et 1
VALIDATION FONCTIONNELLE DU PIPELINE : OK


## 11. Exécution des tests automatisés

Le fichier `tests/test_pipeline.py` vérifie notamment :

- une entrée valide ;
- une colonne obligatoire absente ;
- la classe prédite ;
- la validité des probabilités ;
- la gestion des valeurs manquantes ;
- un DataFrame vide ;
- la prédiction par lot ;
- la shape standardisée ;
- l'absence de valeurs manquantes après preprocessing.

In [14]:
command = [
    sys.executable,
    "-m",
    "pytest",
    "-v",
    str(TEST_DIR / "test_pipeline.py"),
]

completed_process = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)

print(completed_process.stdout)

if completed_process.stderr:
    print(completed_process.stderr)

if completed_process.returncode != 0:
    raise RuntimeError(
        "Les tests automatisés ne passent pas tous."
    )

print("✓ Tous les tests automatisés passent")

============================= test session starts ==============================
platform darwin -- Python 3.9.6, pytest-8.4.2, pluggy-1.6.0 -- /Users/othmanbenmoussa/Desktop/Titanic_Classification/venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/othmanbenmoussa/Desktop/Titanic_Classification
plugins: anyio-4.12.1
collecting ... collected 9 items

tests/test_pipeline.py::test_valid_input PASSED                          [ 11%]
tests/test_pipeline.py::test_missing_column PASSED                       [ 22%]
tests/test_pipeline.py::test_prediction PASSED                           [ 33%]
tests/test_pipeline.py::test_prediction_probability PASSED               [ 44%]
tests/test_pipeline.py::test_missing_values PASSED                       [ 55%]
tests/test_pipeline.py::test_empty_dataframe PASSED                      [ 66%]
tests/test_pipeline.py::test_batch_prediction PASSED                     [ 77%]
tests/test_pipeline.py::test_scaled_output_shape PASSED                  [ 88%]
tes

## 12. Limites actuelles et prochaines étapes

Le pipeline est maintenant reproductible et testé, mais un déploiement complet demanderait encore :

- une API de prédiction, par exemple avec FastAPI ;
- une validation des requêtes avec Pydantic ;
- une conteneurisation Docker ;
- une intégration continue exécutant automatiquement `pytest` ;
- un système de logs ;
- un suivi de la dérive des données et des performances ;
- un versionnement synchronisé du modèle, du scaler et des paramètres de preprocessing.

## 13. Conclusion générale

Ce notebook a validé l'exploitation opérationnelle du modèle final sans réentraîner le système. Les données brutes sont contrôlées, nettoyées, enrichies, encodées et standardisées selon les mêmes règles que pendant l'entraînement.

Les valeurs manquantes sont remplacées à partir des statistiques apprises uniquement sur le jeu d'entraînement. Le scaler sauvegardé est réutilisé avec `transform()` et le modèle retourne une classe ainsi qu'une probabilité de survie.

Les scénarios nominal, incomplet, batch et invalide ont été contrôlés. Les tests automatisés confirment également la robustesse du pipeline. Le projet dispose donc désormais d'une base sérieuse, reproductible et prête à être intégrée dans une API ou une application.